# Modules du pipeline — fichiers sources

Ce notebook regroupe **tous les fichiers `.py` du pipeline** en un seul fichier
facile à envoyer. Chaque cellule ci-dessous correspond à **un fichier**, et
utilise la magie Jupyter `%%writefile` : quand vous **exécutez une cellule**,
elle écrit le fichier `.py` correspondant sur le disque, dans le même dossier
que ce notebook.

**Mode d'emploi :**
1. Placez ce notebook (`modules.ipynb`) et `pipeline.ipynb` dans le même dossier.
2. Exécutez **toutes les cellules de ce notebook** (Run All) — cela recrée les
   5 fichiers `.py` nécessaires au pipeline.
3. Ouvrez et exécutez ensuite `pipeline.ipynb` normalement.

Vous pouvez aussi ignorer l'exécution et simplement **lire/copier le contenu**
de chaque cellule si vous préférez recréer les fichiers manuellement.

| Fichier | Rôle |
|---|---|
| `config.py` | Mots-clés EN/FR, poids et seuils. Modifiez les règles de détection ici. |
| `text_extraction.py` | Extraction de texte : natif (PDF texte), OCR (PDF scannés), hybride. Remplacez le moteur OCR ici. |
| `relevance.py` | Scoring de pertinence : calcule 2 méthodes en parallèle (densité seule / mot-clé + densité) pour comparaison. |
| `pdf_processor.py` | Traite un PDF ou un dossier entier, écrit un PDF filtré par méthode demandée (comparaison possible). |
| `report.py` | Génère le fichier Excel global (onglets Résumé + Détail_pages, avec les 2 méthodes côte à côte). |
| `density_tester.py` | Outil de test unitaire : affiche pour chaque page la densité + le résultat des 2 méthodes, pour ajuster les seuils. |

### `config.py`

Mots-clés EN/FR, poids et seuils. Modifiez les règles de détection ici.

In [ ]:
%%writefile config.py
"""
config.py
---------
Configuration centrale du pipeline : mots-clés (EN/FR) et paramètres par défaut.

Toute modification des règles de détection (nouveaux mots-clés, nouveaux poids,
nouveaux seuils) se fait UNIQUEMENT ici, sans toucher au reste du code.
"""

# ---------------------------------------------------------------------------
# 1. MOTS-CLES / EXPRESSIONS A RECHERCHER
# ---------------------------------------------------------------------------
# Chaque statement financier est une liste de variantes (le mot "consolidated"
# / "consolidé" est ignoré volontairement, comme demandé dans les guidelines).

KEYWORDS_EN = {
    "income_statement": [
        "income statement",
        "profit and loss statement",
        "statement of earnings",
        "statement of operations",
        "statement of results",
    ],
    "balance_sheet": [
        "balance sheet",
        "statement of financial position",
        "statement of assets and liabilities",
        "statement of financial condition",
    ],
    "cash_flow_statement": [
        "cash flow statement",
        "statement of cash flows",
        "cash flow report",
        "statement of cash inflows and outflows",
    ],
    "statement_changes_equity": [
        "statement of changes in equity",
        "statement of shareholders equity",
        "statement of shareholders' equity",
        "statement of owners equity",
        "equity movement statement",
        "statement of equity",
    ],
    "statement_comprehensive_income": [
        "statement of comprehensive income",
        "comprehensive income statement",
        "statement of total comprehensive income",
        "statement of comprehensive earnings",
    ],
}

KEYWORDS_FR = {
    "compte_de_resultat": [
        "compte de resultat",
        "etat du resultat",
        "etat des resultats",
        "compte de pertes et profits",
        "etat des gains et pertes",
    ],
    "bilan": [
        "bilan",
        "bilan comptable",
        "etat de la situation financiere",
        "etat du bilan",
        "bilan des actifs et passifs",
    ],
    "tableau_flux_tresorerie": [
        "tableau des flux de tresorerie",
        "etat des flux de tresorerie",
        "flux de tresorerie",
        "etat des mouvements de tresorerie",
        "releve des flux de tresorerie",
    ],
    "tableau_variations_capitaux_propres": [
        "etat des variations des capitaux propres",
        "tableau des mouvements de capitaux propres",
        "etat des variations du patrimoine net",
        "etat des changements de capitaux propres",
        "tableau des variations de l'equite",
        "tableau des variations de l equite",
    ],
    "etat_resultat_global": [
        "etat du resultat global",
        "compte de resultat global",
        "etat du revenu global",
        "releve du resultat global",
        "tableau du resultat global",
    ],
}

# Mot à ignorer dans la comparaison (on le retire du texte avant de matcher,
# afin qu'il n'influence ni positivement ni négativement la détection).
IGNORED_WORDS = ["consolidated", "consolide", "consolidée", "consolidées", "consolidés"]


# ---------------------------------------------------------------------------
# 2. PARAMETRES PAR DEFAUT DU PIPELINE
# ---------------------------------------------------------------------------

DEFAULT_SETTINGS = {
    # --- OCR / extraction de texte ---
    "ocr_lang": "eng+fra",          # langues tesseract
    "ocr_dpi": 300,                  # résolution de rasterisation pour l'OCR
    "ocr_psm": None,                 # Page Segmentation Mode tesseract (None = auto).
                                      # Essayez 6 (bloc de texte uniforme) ou 4
                                      # (colonne de texte) si l'OCR lit mal vos tableaux.

    # Stratégie d'extraction : "hybrid" (natif puis OCR en secours - recommandé),
    # "ocr_only" (force l'OCR sur TOUTES les pages, natif ou pas - utile si vos
    # PDF ont un problème d'encodage de police qui rend le mode hybride peu
    # fiable), "native_only" (jamais d'OCR, PDF 100% texte uniquement).
    "extraction_mode": "hybrid",

    # nombre minimal de caractères "utiles" extraits nativement en dessous
    # duquel on considère la page comme scannée et on bascule sur l'OCR
    "native_text_min_chars": 40,
    # proportion minimale de caractères "normaux" (lettres/chiffres/espaces/
    # ponctuation courante) exigée pour faire confiance au texte natif ; en
    # dessous, on suppose un problème d'encodage de police et on bascule sur
    # l'OCR même si le texte natif est assez long (cause fréquente de pages
    # ratées quand le mode hybride "a l'air de marcher" mais lit du charabia)
    "native_min_valid_char_ratio": 0.85,

    # --- scoring de pertinence ---
    # DEUX MÉTHODES DE DÉCISION sont toujours calculées pour chaque page,
    # afin de pouvoir les comparer :
    #   - "density_only"        : pertinente si la densité numérique (+
    #     zero-shot éventuel) dépasse le seuil, QUE le mot-clé matche ou non.
    #   - "keyword_and_density" : pertinente seulement si un mot-clé matche
    #     ET que la densité numérique confirme (mot-clé = condition
    #     d'entrée obligatoire, densité = confirmation).
    #
    # `methods` définit lesquelles sont utilisées pour FILTRER les PDF et
    # produire un/des PDF de sortie. Par défaut, LES DEUX sont actives :
    # deux PDF filtrés sont produits par document source (dans des
    # sous-dossiers output_pdfs/density_only/ et output_pdfs/keyword_and_density/),
    # et le rapport Excel montre les deux résultats côte à côte pour
    # comparaison. Passez une liste à un seul élément pour n'en utiliser
    # qu'une (le PDF filtré est alors écrit directement dans output_pdfs/).
    "methods": ["density_only", "keyword_and_density"],

    # tolère les erreurs d'OCR qui cassent une expression à plusieurs mots
    # (ex: "balance sheet" mal reconnu à cause d'un caractère parasite) en
    # acceptant un match si tous les mots significatifs de l'expression sont
    # présents séparément dans le texte, même non adjacents
    "fuzzy_word_presence": True,

    # poids relatifs entre densité numérique et zero-shot dans le score de
    # confirmation (somme libre, normalisés automatiquement ; le poids
    # zero-shot est ignoré si use_zero_shot=False)
    "weight_numeric_density": 0.7,
    "weight_zero_shot": 0.3,

    # seuil de densité numérique (proportion de caractères numériques /
    # caractères totaux) au-delà duquel on considère la page "riche en chiffres"
    "numeric_density_threshold": 0.06,

    # nombre ABSOLU minimal de chiffres exigé sur la page (évite qu'un texte
    # court avec juste une numérotation "1. 2. 3." soit pris pour une page
    # dense en chiffres à cause du seul ratio)
    "numeric_min_digit_count": 15,

    # seuil de CONFIRMATION appliqué au score de densité numérique (+ zero-shot
    # éventuel), que le mot-clé ait matché ou non. En dessous, la page est
    # rejetée.
    "confirmation_threshold": 0.5,

    # --- détection de pages de continuation ---
    # Un bilan/compte de résultat s'étale souvent sur plusieurs pages, mais
    # le titre n'apparaît qu'en haut de la première. Si activé, une page qui
    # suit immédiatement une page pertinente, sans mot-clé propre mais avec
    # une densité numérique suffisante, est considérée comme la suite du
    # même tableau et gardée aussi. Avec `require_keyword_match=False`
    # (par défaut), ce mécanisme est redondant dans la plupart des cas
    # (la densité seule suffit déjà), mais reste utile en filet de sécurité.
    "enable_continuation_detection": True,
    "continuation_threshold": 0.6,

    # --- traitement de gros volumes (beaucoup de PDF / documents longs) ---
    # Reprise après plantage : après chaque PDF traité, une sauvegarde est
    # écrite dans <output_dir>/.pipeline_checkpoint.json. Si le pipeline est
    # interrompu (plantage, coupure), relancez simplement la même commande :
    # les PDF déjà traités sont automatiquement ignorés et on reprend où on
    # s'était arrêté.
    "enable_checkpoint": True,
    "checkpoint_path": None,  # None = <output_dir>/.pipeline_checkpoint.json

    # Nombre de PDF traités EN PARALLÈLE (1 = séquentiel). Augmentez une fois
    # les réglages validés sur vos documents (ex: os.cpu_count() - 1) pour
    # accélérer sur de gros lots. Incompatible avec use_zero_shot=True
    # (retombe automatiquement en séquentiel dans ce cas).
    "parallel_workers": 1,

    # Si True (par défaut), un fichier qui a échoué lors d'un run précédent
    # est retenté au run suivant (utile si vous corrigez/remplacez le
    # fichier source). Seuls les fichiers traités AVEC SUCCÈS sont
    # définitivement ignorés lors d'une reprise.
    "retry_errors": True,

    # activer/désactiver le classifieur zero-shot (nécessite `transformers`+`torch`)
    "use_zero_shot": False,
    "zero_shot_model": "joeddav/xlm-roberta-large-xnli",  # multilingue EN/FR
    "zero_shot_labels": [
        "financial statement page (income statement, balance sheet, cash flow, equity)",
        "irrelevant page (cover page, table of contents, notes, disclaimer, appendix)",
    ],
}


### `text_extraction.py`

Extraction de texte : natif (PDF texte), OCR (PDF scannés), hybride. Remplacez le moteur OCR ici.

In [ ]:
%%writefile text_extraction.py
"""
text_extraction.py
-------------------
Module responsable UNIQUEMENT d'extraire le texte de chaque page d'un PDF.

Architecture volontairement modulaire :
- `TextExtractor` est une interface abstraite. Le reste du pipeline (relevance,
  pdf_processor) ne dépend QUE de cette interface (méthode `extract_page_texts`).
- On peut donc remplacer le moteur OCR (Tesseract -> EasyOCR, PaddleOCR, une
  API cloud, etc.) en écrivant une nouvelle classe qui hérite de
  `TextExtractor`, SANS RIEN CHANGER ailleurs dans le pipeline.

Trois implémentations, plus une fabrique `build_extractor(settings)` :
1. NativeTextExtractor  : lit le texte déjà présent dans le PDF (pdfplumber).
   Rapide, mais peut être trompeur sur certains PDF "normaux" dont
   l'encodage de police est défectueux (texte extrait = charabia).
2. TesseractOCRExtractor: rasterise chaque page (pdf2image/poppler) puis
   applique un OCR (pytesseract/Tesseract). Fonctionne sur les PDF scannés,
   et peut aussi être utilisé en mode "tout OCR" si l'extraction native
   n'est pas fiable sur vos documents.
3. HybridTextExtractor  : essaie d'abord l'extraction native page par page ;
   bascule sur l'OCR si le texte natif est trop court OU trop "sale"
   (ratio de caractères normaux trop faible -> probable problème
   d'encodage de police).

`build_extractor(settings)` choisit l'implémentation à utiliser selon
`settings["extraction_mode"]` ("hybrid" / "ocr_only" / "native_only"),
pratique pour changer de stratégie sans toucher au notebook.
"""

from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Dict, List, Optional

import pdfplumber
import pytesseract
from pdf2image import convert_from_path


@dataclass
class PageText:
    """Texte extrait d'une page, avec un indicateur de la méthode utilisée."""
    page_number: int          # 1-indexed
    text: str
    method: str               # "native" ou "ocr" (ou autre nom d'implémentation)


class TextExtractor(ABC):
    """Interface commune à tout moteur d'extraction de texte."""

    @abstractmethod
    def extract_page_texts(self, pdf_path: str) -> List[PageText]:
        """Retourne une liste de PageText, une entrée par page du PDF."""
        raise NotImplementedError


# ---------------------------------------------------------------------------
# 1. Extraction native (PDF "normaux", texte déjà encodé)
# ---------------------------------------------------------------------------
class NativeTextExtractor(TextExtractor):
    def extract_page_texts(self, pdf_path: str) -> List[PageText]:
        results: List[PageText] = []
        with pdfplumber.open(pdf_path) as pdf:
            for i, page in enumerate(pdf.pages, start=1):
                text = page.extract_text() or ""
                results.append(PageText(page_number=i, text=text, method="native"))
        return results


# ---------------------------------------------------------------------------
# 2. Extraction OCR (PDF scannés / images, ou mode "tout OCR" forcé)
#    -> C'est CETTE classe qu'il faut remplacer si vous changez de moteur OCR
#       (ex: EasyOCR, PaddleOCR, Google Vision, Azure Document Intelligence...).
#       Il suffit de respecter l'interface TextExtractor.
# ---------------------------------------------------------------------------
class TesseractOCRExtractor(TextExtractor):
    def __init__(self, lang: str = "eng+fra", dpi: int = 300, psm: Optional[int] = None):
        self.lang = lang
        self.dpi = dpi
        # Page Segmentation Mode de Tesseract. None = comportement par défaut
        # (psm 3, segmentation automatique complète). psm 6 ("bloc de texte
        # uniforme") ou psm 4 ("colonne de texte de tailles variables")
        # peuvent améliorer la lecture de tableaux financiers denses selon
        # vos documents - à tester sur vos PDF.
        self.psm = psm

    def _tesseract_config(self) -> str:
        return f"--psm {self.psm}" if self.psm is not None else ""

    def extract_page_texts(self, pdf_path: str) -> List[PageText]:
        images = convert_from_path(pdf_path, dpi=self.dpi)
        results: List[PageText] = []
        for i, img in enumerate(images, start=1):
            text = pytesseract.image_to_string(
                img, lang=self.lang, config=self._tesseract_config()
            )
            results.append(PageText(page_number=i, text=text, method="ocr"))
        return results

    def extract_single_page_text(self, pdf_path: str, page_number: int) -> str:
        """OCR d'une seule page (1-indexed) - utilisé par HybridTextExtractor
        pour éviter de re-rasteriser tout le PDF quand une seule page a besoin d'OCR."""
        images = convert_from_path(
            pdf_path, dpi=self.dpi, first_page=page_number, last_page=page_number
        )
        if not images:
            return ""
        return pytesseract.image_to_string(
            images[0], lang=self.lang, config=self._tesseract_config()
        )


# ---------------------------------------------------------------------------
# 3. Extracteur hybride : natif d'abord, OCR en secours page par page.
#    Deux raisons de basculer sur l'OCR pour une page donnée :
#      (a) le texte natif est trop court (page probablement scannée)
#      (b) le texte natif est assez long MAIS contient trop de caractères
#          "anormaux" (souvent le signe d'un problème d'encodage de police
#          dans le PDF -> le texte natif est du charabia, même s'il fait
#          plusieurs centaines de caractères). C'est une cause fréquente de
#          faux négatifs quand l'extraction native "a l'air de marcher"
#          mais renvoie en fait un texte inexploitable.
# ---------------------------------------------------------------------------
class HybridTextExtractor(TextExtractor):
    def __init__(
        self,
        native_extractor: TextExtractor | None = None,
        ocr_extractor: TesseractOCRExtractor | None = None,
        native_text_min_chars: int = 40,
        min_valid_char_ratio: float = 0.85,
    ):
        self.native_extractor = native_extractor or NativeTextExtractor()
        self.ocr_extractor = ocr_extractor or TesseractOCRExtractor()
        self.native_text_min_chars = native_text_min_chars
        # proportion minimale de caractères "normaux" (lettres, chiffres,
        # espaces, ponctuation courante) exigée pour faire confiance au
        # texte natif ; en dessous, on suppose un problème d'encodage et on
        # bascule sur l'OCR même si le texte natif est assez long.
        self.min_valid_char_ratio = min_valid_char_ratio

    def _looks_valid(self, text: str) -> bool:
        stripped = text.strip()
        if len(stripped) < self.native_text_min_chars:
            return False
        normal_chars = sum(
            1 for c in stripped
            if c.isalnum() or c.isspace() or c in ".,;:%()-/€$'\""
        )
        ratio = normal_chars / len(stripped)
        return ratio >= self.min_valid_char_ratio

    def extract_page_texts(self, pdf_path: str) -> List[PageText]:
        native_pages = self.native_extractor.extract_page_texts(pdf_path)
        final_pages: List[PageText] = []

        for page in native_pages:
            if self._looks_valid(page.text):
                final_pages.append(page)
            else:
                # Page probablement scannée OU texte natif corrompu -> OCR ciblé
                ocr_text = self.ocr_extractor.extract_single_page_text(
                    pdf_path, page.page_number
                )
                final_pages.append(
                    PageText(page_number=page.page_number, text=ocr_text, method="ocr")
                )
        return final_pages


# ---------------------------------------------------------------------------
# Fabrique : construit l'extracteur à utiliser selon settings["extraction_mode"]
# ---------------------------------------------------------------------------
def build_extractor(settings: Dict) -> TextExtractor:
    mode = settings.get("extraction_mode", "hybrid")
    ocr_extractor = TesseractOCRExtractor(
        lang=settings.get("ocr_lang", "eng+fra"),
        dpi=settings.get("ocr_dpi", 300),
        psm=settings.get("ocr_psm"),
    )

    if mode == "ocr_only":
        # Force l'OCR sur TOUTES les pages, natif ou pas. Recommandé si vos
        # documents ont des soucis d'extraction native (encodage de police
        # défectueux) qui rendent le mode hybride peu fiable.
        return ocr_extractor
    elif mode == "native_only":
        return NativeTextExtractor()
    else:
        return HybridTextExtractor(
            native_extractor=NativeTextExtractor(),
            ocr_extractor=ocr_extractor,
            native_text_min_chars=settings.get("native_text_min_chars", 40),
            min_valid_char_ratio=settings.get("native_min_valid_char_ratio", 0.85),
        )


### `relevance.py`

Scoring de pertinence : calcule 2 méthodes en parallèle (densité seule / mot-clé + densité) pour comparaison.

In [ ]:
%%writefile relevance.py
"""
relevance.py
------------
Module responsable UNIQUEMENT de juger si le texte d'une page est "pertinent"
(page de statement financier) ou non.

Comme pour l'extraction, chaque méthode de scoring est une classe qui respecte
l'interface `RelevanceScorer` (méthode `score(text) -> float` dans [0, 1]).
Le `CompositeScorer` les combine par une moyenne pondérée (poids définis dans
config.py). On peut activer/désactiver ou remplacer un scorer sans toucher
au reste du pipeline.
"""

from __future__ import annotations

import re
import unicodedata
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Dict, List, Optional

from config import (
    KEYWORDS_EN,
    KEYWORDS_FR,
    IGNORED_WORDS,
    DEFAULT_SETTINGS,
)


def _normalize(text: str) -> str:
    """Minuscule, sans accents, espaces multiples réduits - pour un matching robuste
    (OCR renvoie parfois des accents mal reconnus, ou casse un mot en fin de ligne)."""
    text = text.lower()
    # fusionne les mots coupés en fin de ligne par un tiret ("bi-\nlan" -> "bilan"),
    # cas fréquent en sortie d'OCR sur des colonnes de texte étroites
    text = re.sub(r"-\s*\n\s*", "", text)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    for w in IGNORED_WORDS:
        w_norm = unicodedata.normalize("NFKD", w.lower())
        w_norm = "".join(c for c in w_norm if not unicodedata.combining(c))
        text = text.replace(w_norm, " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


@dataclass
class RelevanceResult:
    score: float
    method_results: Dict[str, bool]  # ex: {"density_only": True, "keyword_and_density": False}
    matched_keywords: List[str] = field(default_factory=list)
    details: Dict[str, float] = field(default_factory=dict)


class RelevanceScorer(ABC):
    @abstractmethod
    def score(self, text: str) -> float:
        """Retourne un score dans [0, 1]."""
        raise NotImplementedError


# ---------------------------------------------------------------------------
# 1. Scorer par mots-clés (règle 1 des guidelines)
# ---------------------------------------------------------------------------
class KeywordScorer(RelevanceScorer):
    def __init__(
        self,
        keywords_en: Dict[str, List[str]] = None,
        keywords_fr: Dict[str, List[str]] = None,
        fuzzy_word_presence: bool = True,
    ):
        self.keywords_en = keywords_en or KEYWORDS_EN
        self.keywords_fr = keywords_fr or KEYWORDS_FR
        # si True, une expression à plusieurs mots (ex: "balance sheet") est
        # aussi considérée trouvée si tous ses mots significatifs apparaissent
        # séparément dans le texte, même non adjacents. Tolère les erreurs
        # d'OCR qui insèrent un espace ou un caractère parasite au milieu
        # d'une expression, cause fréquente de mots-clés manqués.
        self.fuzzy_word_presence = fuzzy_word_presence

        # à plat, normalisées une fois pour toutes
        self._all_terms = []       # liste de termes normalisés (phrase complète)
        self._all_terms_words = [] # liste parallèle : mots significatifs de chaque terme
        for group in list(self.keywords_en.values()) + list(self.keywords_fr.values()):
            for term in group:
                norm_term = _normalize(term)
                self._all_terms.append(norm_term)
                self._all_terms_words.append(norm_term.split())

    def matched_terms(self, text: str) -> List[str]:
        norm_text = _normalize(text)
        matches = []

        for term in self._all_terms:
            if term in norm_text:
                matches.append(term)

        if self.fuzzy_word_presence:
            # utilise des frontières de mots pour éviter qu'un mot ne matche
            # comme sous-chaîne d'un autre mot plus long
            text_words = set(re.findall(r"[a-z0-9]+", norm_text))
            for term, words in zip(self._all_terms, self._all_terms_words):
                if term in matches:
                    continue
                # ne s'applique qu'aux expressions à 2+ mots significatifs
                # (les mots-clés d'un seul mot, ex: "bilan", sont déjà
                # couverts par le matching exact ci-dessus)
                if len(words) >= 2 and all(w in text_words for w in words):
                    matches.append(f"{term} (approx.)")

        return matches

    def score(self, text: str) -> float:
        # dès qu'un terme matche (exact ou approché), la page passe la
        # première étape (voir CompositeScorer : le mot-clé seul ne suffit
        # PAS à garder la page, il ouvre juste la porte à la confirmation
        # par la densité numérique).
        return 1.0 if self.matched_terms(text) else 0.0


# ---------------------------------------------------------------------------
# 2. Scorer de densité numérique (règle 2 des guidelines)
# ---------------------------------------------------------------------------
class NumericDensityScorer(RelevanceScorer):
    def __init__(self, threshold: float = 0.06, min_digit_count: int = 15):
        self.threshold = threshold
        # Nombre ABSOLU minimal de chiffres exigé sur la page. Sans ce
        # garde-fou, un texte très court avec juste une numérotation
        # ("1. Rapport 2. Bilan 3. Annexes") peut afficher une densité
        # (ratio) élevée alors qu'il n'y a presque aucun vrai chiffre
        # financier. Une vraie page de bilan/compte de résultat contient
        # typiquement des dizaines de chiffres.
        self.min_digit_count = min_digit_count

    def digit_count(self, text: str) -> int:
        return sum(c.isdigit() for c in text) if text else 0

    def density(self, text: str) -> float:
        if not text:
            return 0.0
        return self.digit_count(text) / max(len(text), 1)

    def score(self, text: str) -> float:
        d = self.density(text)
        count = self.digit_count(text)

        # score basé sur le ratio (densité), normalisé au seuil configuré
        density_score = min(d / self.threshold, 1.0) if self.threshold > 0 else 0.0
        # score basé sur le volume absolu de chiffres
        count_score = (
            min(count / self.min_digit_count, 1.0) if self.min_digit_count > 0 else 1.0
        )

        # les DEUX conditions doivent être satisfaites (on prend le minimum) :
        # une page doit être à la fois dense en chiffres ET en contenir
        # suffisamment en valeur absolue pour être un vrai tableau financier.
        return min(density_score, count_score)


# ---------------------------------------------------------------------------
# 3. Scorer zero-shot (règle 3, optionnel - nécessite `transformers` + `torch`)
# ---------------------------------------------------------------------------
class ZeroShotScorer(RelevanceScorer):
    """Scorer optionnel basé sur un modèle de classification zero-shot local
    (Hugging Face). Désactivé par défaut (use_zero_shot=False dans config.py)
    car il nécessite `pip install transformers torch` et un téléchargement de
    modèle. Peut être remplacé par n'importe quel modèle zero-shot compatible
    `pipeline("zero-shot-classification", ...)`.
    """

    def __init__(self, model_name: str, candidate_labels: List[str]):
        try:
            from transformers import pipeline
        except ImportError as e:
            raise ImportError(
                "ZeroShotScorer nécessite `transformers` et `torch`. "
                "Installez-les avec: pip install transformers torch"
            ) from e
        self._classifier = pipeline("zero-shot-classification", model=model_name)
        self.candidate_labels = candidate_labels
        # on suppose que le premier label de la liste = label "pertinent"
        self.relevant_label = candidate_labels[0]

    def score(self, text: str) -> float:
        if not text.strip():
            return 0.0
        # on tronque le texte pour rester dans la limite de tokens du modèle
        truncated = text[:2000]
        result = self._classifier(truncated, self.candidate_labels)
        label_scores = dict(zip(result["labels"], result["scores"]))
        return float(label_scores.get(self.relevant_label, 0.0))


# ---------------------------------------------------------------------------
# 4. Scorer composite : calcule DEUX MÉTHODES DE DÉCISION EN PARALLÈLE pour
#    chaque page, afin de pouvoir les comparer :
#
#    - "density_only"          : pertinente si la densité numérique (+
#      zero-shot éventuel) dépasse le seuil, QUE le mot-clé ait matché ou non.
#    - "keyword_and_density"    : pertinente seulement si un mot-clé a
#      matché ET que la densité numérique dépasse le seuil (mot-clé =
#      condition d'entrée obligatoire, densité = confirmation).
#
#    Les deux résultats sont toujours calculés ; c'est `pdf_processor.py`
#    (via `settings["methods"]`) qui décide lesquelles utiliser pour filtrer
#    les PDF (une seule, ou les deux pour comparaison côte à côte).
# ---------------------------------------------------------------------------
METHOD_DENSITY_ONLY = "density_only"
METHOD_KEYWORD_AND_DENSITY = "keyword_and_density"
ALL_METHODS = [METHOD_DENSITY_ONLY, METHOD_KEYWORD_AND_DENSITY]


class CompositeScorer:
    def __init__(
        self,
        keyword_scorer: KeywordScorer,
        numeric_scorer: NumericDensityScorer,
        zero_shot_scorer: Optional[ZeroShotScorer] = None,
        weight_numeric: float = 0.7,
        weight_zero_shot: float = 0.3,
        confirmation_threshold: float = 0.5,
    ):
        self.keyword_scorer = keyword_scorer
        self.numeric_scorer = numeric_scorer
        self.zero_shot_scorer = zero_shot_scorer
        self.confirmation_threshold = confirmation_threshold

        # normalisation des poids entre densité numérique et zero-shot
        # (ignore le poids zero-shot si le scorer est absent)
        w_zs = weight_zero_shot if zero_shot_scorer is not None else 0.0
        total = weight_numeric + w_zs
        total = total if total > 0 else 1.0
        self.weight_numeric = weight_numeric / total
        self.weight_zero_shot = w_zs / total

    def evaluate(self, text: str) -> RelevanceResult:
        matched = self.keyword_scorer.matched_terms(text)
        kw_score = 1.0 if matched else 0.0
        num_score = self.numeric_scorer.score(text)
        zs_score = self.zero_shot_scorer.score(text) if self.zero_shot_scorer else 0.0

        # score de confirmation : moyenne pondérée densité numérique + zero-shot
        confirmation_score = (
            self.weight_numeric * num_score + self.weight_zero_shot * zs_score
        )
        density_ok = confirmation_score >= self.confirmation_threshold

        method_results = {
            METHOD_DENSITY_ONLY: density_ok,
            METHOD_KEYWORD_AND_DENSITY: bool(matched) and density_ok,
        }

        return RelevanceResult(
            score=confirmation_score,
            method_results=method_results,
            matched_keywords=matched,
            details={
                "keyword_score": kw_score,
                "numeric_density_score": num_score,
                "zero_shot_score": zs_score,
                "confirmation_score": confirmation_score,
            },
        )


def build_default_composite_scorer(settings: Dict = None) -> CompositeScorer:
    """Factory pratique : construit un CompositeScorer à partir de config.DEFAULT_SETTINGS
    (ou d'un dict de settings personnalisé)."""
    s = settings or DEFAULT_SETTINGS
    keyword_scorer = KeywordScorer(
        fuzzy_word_presence=s.get("fuzzy_word_presence", True)
    )
    numeric_scorer = NumericDensityScorer(
        threshold=s["numeric_density_threshold"],
        min_digit_count=s.get("numeric_min_digit_count", 15),
    )

    zero_shot_scorer = None
    if s.get("use_zero_shot"):
        zero_shot_scorer = ZeroShotScorer(
            model_name=s["zero_shot_model"],
            candidate_labels=s["zero_shot_labels"],
        )

    return CompositeScorer(
        keyword_scorer=keyword_scorer,
        numeric_scorer=numeric_scorer,
        zero_shot_scorer=zero_shot_scorer,
        weight_numeric=s["weight_numeric_density"],
        weight_zero_shot=s["weight_zero_shot"],
        confirmation_threshold=s["confirmation_threshold"],
    )


### `pdf_processor.py`

Traite un PDF ou un dossier entier, écrit un PDF filtré par méthode demandée (comparaison possible).

In [ ]:
%%writefile pdf_processor.py
"""
pdf_processor.py
----------------
Orchestration au niveau d'UN SEUL PDF, en 2 passages :
1. Extraire le texte de chaque page (via un TextExtractor) et évaluer
   chaque page (via un CompositeScorer), qui calcule TOUJOURS les deux
   méthodes de décision en parallèle :
   - "density_only"       : densité numérique seule
   - "keyword_and_density" : mot-clé obligatoire + densité de confirmation
2. Pour chaque méthode demandée (`settings["methods"]`), repasser sur les
   pages pour détecter les PAGES DE CONTINUATION (un tableau qui continue
   sur la page suivante sans répéter le titre), puis écrire un PDF filtré
   séparé par méthode.

Si UNE SEULE méthode est demandée, le PDF filtré est écrit directement dans
`output_dir`. Si LES DEUX sont demandées, chaque méthode écrit dans son
propre sous-dossier (`output_dir/density_only/`, `output_dir/keyword_and_density/`)
pour pouvoir comparer facilement les deux résultats côte à côte.

Ce module ne connaît ni le moteur OCR précis, ni le détail des règles de
scoring : il dépend uniquement des interfaces `TextExtractor` et
`CompositeScorer`. On peut donc changer d'OCR ou de logique de scoring sans
modifier une seule ligne ici.

Pour les GROS LOTS (beaucoup de PDF ou documents très longs), deux
mécanismes supplémentaires sont fournis au niveau du dossier
(`process_folder`) :
- REPRISE APRÈS PLANTAGE : une sauvegarde (`.pipeline_checkpoint.json`) est
  écrite après chaque PDF traité.
- TRAITEMENT EN PARALLÈLE (optionnel) : plusieurs PDF peuvent être traités
  simultanément sur plusieurs cœurs CPU (`settings["parallel_workers"]`).
"""

from __future__ import annotations

import json
import os
import time
from dataclasses import asdict, dataclass, field
from typing import Dict, List, Optional, Tuple

from pypdf import PdfReader, PdfWriter

from text_extraction import TextExtractor
from relevance import CompositeScorer, RelevanceResult, METHOD_DENSITY_ONLY, METHOD_KEYWORD_AND_DENSITY

DEFAULT_METHODS = [METHOD_DENSITY_ONLY, METHOD_KEYWORD_AND_DENSITY]


@dataclass
class PageResult:
    page_number: int           # 1-indexed
    score: float
    matched_keywords: List[str]
    extraction_method: str
    method_results: Dict[str, bool] = field(default_factory=dict)
    continuation_by_method: Dict[str, bool] = field(default_factory=dict)


@dataclass
class PdfProcessingResult:
    pdf_name: str
    input_path: str
    total_pages: int = 0
    output_paths_by_method: Dict[str, str] = field(default_factory=dict)
    relevant_pages_by_method: Dict[str, List[int]] = field(default_factory=dict)
    page_results: List[PageResult] = field(default_factory=list)
    error: str | None = None


def _detect_continuation_pages(
    page_results: List[PageResult],
    method: str,
    continuation_threshold: float,
) -> None:
    """Modifie page_results EN PLACE, pour la méthode donnée : marque comme
    pertinente toute page qui suit immédiatement une page pertinente, n'a
    trouvé aucun mot-clé, mais a un score de densité numérique suffisant
    pour être la suite du même tableau financier."""
    for i in range(1, len(page_results)):
        prev, curr = page_results[i - 1], page_results[i]
        if (
            prev.method_results.get(method, False)
            and not curr.method_results.get(method, False)
            and not curr.matched_keywords
            and curr.score >= continuation_threshold
        ):
            curr.method_results[method] = True
            curr.continuation_by_method[method] = True


def process_single_pdf(
    pdf_path: str,
    extractor: TextExtractor,
    scorer: CompositeScorer,
    output_dir: str,
    settings: Optional[Dict] = None,
) -> PdfProcessingResult:
    """Traite un seul PDF: évalue chaque page pour chaque méthode demandée,
    applique la détection de continuation, puis écrit un PDF filtré par
    méthode.

    Si aucune page n'est jugée pertinente pour une méthode donnée, aucun
    fichier de sortie n'est écrit pour cette méthode (mais le résultat est
    quand même reporté, utile pour le suivi dans le rapport Excel).
    """
    settings = settings or {}
    methods = settings.get("methods", DEFAULT_METHODS)
    enable_continuation = settings.get("enable_continuation_detection", True)
    continuation_threshold = settings.get("continuation_threshold", 0.6)
    multi_method = len(methods) > 1

    pdf_name = os.path.basename(pdf_path)
    result = PdfProcessingResult(pdf_name=pdf_name, input_path=pdf_path)

    try:
        page_texts = extractor.extract_page_texts(pdf_path)
        result.total_pages = len(page_texts)

        # --- Passage 1 : évaluation indépendante de chaque page (les deux
        # méthodes sont toujours calculées par scorer.evaluate()) ---
        page_results: List[PageResult] = []
        for page_text in page_texts:
            evaluation: RelevanceResult = scorer.evaluate(page_text.text)
            page_results.append(
                PageResult(
                    page_number=page_text.page_number,
                    score=evaluation.score,
                    matched_keywords=evaluation.matched_keywords,
                    extraction_method=page_text.method,
                    method_results=dict(evaluation.method_results),
                )
            )

        # --- Passage 2 : continuation de tableau, par méthode demandée ---
        if enable_continuation:
            for method in methods:
                _detect_continuation_pages(page_results, method, continuation_threshold)

        result.page_results = page_results

        # --- Passage 3 : un PDF filtré + une liste de pages PAR MÉTHODE ---
        reader = None
        for method in methods:
            relevant_pages = [
                pr.page_number for pr in page_results if pr.method_results.get(method)
            ]
            result.relevant_pages_by_method[method] = relevant_pages

            if relevant_pages:
                if reader is None:
                    reader = PdfReader(pdf_path)
                writer = PdfWriter()
                for pr in page_results:
                    if pr.method_results.get(method):
                        writer.add_page(reader.pages[pr.page_number - 1])

                method_output_dir = os.path.join(output_dir, method) if multi_method else output_dir
                os.makedirs(method_output_dir, exist_ok=True)
                output_path = os.path.join(method_output_dir, pdf_name)
                with open(output_path, "wb") as f:
                    writer.write(f)
                result.output_paths_by_method[method] = output_path

    except Exception as e:  # on isole l'erreur pour ne pas casser le traitement du dossier entier
        result.error = f"{type(e).__name__}: {e}"

    return result


# ---------------------------------------------------------------------------
# Sauvegarde intermédiaire (reprise après plantage)
# ---------------------------------------------------------------------------
def _checkpoint_path(output_dir: str, settings: Dict) -> str:
    return settings.get("checkpoint_path") or os.path.join(
        output_dir, ".pipeline_checkpoint.json"
    )


def _load_checkpoint(path: str) -> Dict[str, PdfProcessingResult]:
    if not os.path.exists(path):
        return {}
    try:
        with open(path, "r", encoding="utf-8") as f:
            raw = json.load(f)
    except (json.JSONDecodeError, OSError):
        return {}

    results: Dict[str, PdfProcessingResult] = {}
    for pdf_name, data in raw.items():
        page_results = [PageResult(**pr) for pr in data.get("page_results", [])]
        results[pdf_name] = PdfProcessingResult(
            pdf_name=data["pdf_name"],
            input_path=data["input_path"],
            total_pages=data.get("total_pages", 0),
            output_paths_by_method=data.get("output_paths_by_method", {}),
            relevant_pages_by_method=data.get("relevant_pages_by_method", {}),
            page_results=page_results,
            error=data.get("error"),
        )
    return results


def _save_checkpoint(path: str, results_by_name: Dict[str, PdfProcessingResult]) -> None:
    raw = {name: asdict(res) for name, res in results_by_name.items()}
    tmp_path = path + ".tmp"
    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(raw, f, ensure_ascii=False, indent=2)
    os.replace(tmp_path, path)


# ---------------------------------------------------------------------------
# Traitement d'un dossier entier, avec reprise + parallélisation optionnelle
# ---------------------------------------------------------------------------
def _process_one_file(
    args: Tuple[str, str, TextExtractor, CompositeScorer, str, Optional[Dict]]
) -> Tuple[str, PdfProcessingResult]:
    """Fonction de niveau module (nécessaire pour être "picklable" par
    ProcessPoolExecutor lors du traitement en parallèle)."""
    filename, input_dir, extractor, scorer, output_dir, settings = args
    pdf_path = os.path.join(input_dir, filename)
    return filename, process_single_pdf(pdf_path, extractor, scorer, output_dir, settings)


def process_folder(
    input_dir: str,
    output_dir: str,
    extractor: TextExtractor,
    scorer: CompositeScorer,
    settings: Optional[Dict] = None,
    verbose: bool = True,
) -> List[PdfProcessingResult]:
    """Traite tous les PDF d'un dossier et retourne la liste des résultats
    (un PdfProcessingResult par fichier, dans l'ordre du dossier).

    - `settings["methods"]` : liste des méthodes à utiliser, parmi
      "density_only" et "keyword_and_density". Par défaut, LES DEUX sont
      calculées pour permettre la comparaison. Passez une liste à un seul
      élément pour n'en utiliser qu'une.
    - `settings["enable_checkpoint"]` (True par défaut) : reprise après
      plantage, voir `process_single_pdf`.
    - `settings["parallel_workers"]` > 1 : traitement en parallèle.
    """
    settings = settings or {}
    enable_checkpoint = settings.get("enable_checkpoint", True)
    parallel_workers = settings.get("parallel_workers", 1)

    os.makedirs(output_dir, exist_ok=True)
    pdf_files = sorted(f for f in os.listdir(input_dir) if f.lower().endswith(".pdf"))

    checkpoint_path = _checkpoint_path(output_dir, settings)
    all_checkpointed = _load_checkpoint(checkpoint_path) if enable_checkpoint else {}

    retry_errors = settings.get("retry_errors", True)
    if retry_errors:
        results_by_name = {n: r for n, r in all_checkpointed.items() if not r.error}
    else:
        results_by_name = dict(all_checkpointed)

    todo = [f for f in pdf_files if f not in results_by_name]
    n_skipped = len(pdf_files) - len(todo)
    if verbose and n_skipped:
        print(
            f"{n_skipped} PDF déjà traité(s) trouvé(s) dans la sauvegarde "
            f"({checkpoint_path}), ignoré(s). Reprise sur les {len(todo)} restant(s)."
        )

    if parallel_workers and parallel_workers > 1 and settings.get("use_zero_shot"):
        if verbose:
            print(
                "Zero-shot activé -> traitement en parallèle désactivé "
                "(le modèle IA ne se sérialise pas entre processus). Retour en séquentiel."
            )
        parallel_workers = 1

    def _record(filename: str, res: PdfProcessingResult, index: int, total: int, elapsed: float) -> None:
        results_by_name[filename] = res
        if enable_checkpoint:
            _save_checkpoint(checkpoint_path, results_by_name)
        if verbose:
            if res.error:
                print(f"[{index}/{total}] {filename} -> ERREUR: {res.error} ({elapsed:.1f}s)")
            else:
                parts = []
                for method, pages in res.relevant_pages_by_method.items():
                    n_cont = sum(
                        pr.continuation_by_method.get(method, False) for pr in res.page_results
                    )
                    extra = f", {n_cont} continuation(s)" if n_cont else ""
                    parts.append(f"{method}: {len(pages)}/{res.total_pages}{extra}")
                print(f"[{index}/{total}] {filename} -> " + " | ".join(parts) + f" ({elapsed:.1f}s)")

    if todo:
        start_all = time.time()
        if parallel_workers and parallel_workers > 1:
            import concurrent.futures

            args_list = [(f, input_dir, extractor, scorer, output_dir, settings) for f in todo]
            with concurrent.futures.ProcessPoolExecutor(max_workers=parallel_workers) as executor:
                futures = {executor.submit(_process_one_file, a): a[0] for a in args_list}
                start_times = {a[0]: time.time() for a in args_list}
                for i, future in enumerate(concurrent.futures.as_completed(futures), start=1):
                    filename, res = future.result()
                    _record(filename, res, i, len(todo), time.time() - start_times[filename])
        else:
            for i, filename in enumerate(todo, start=1):
                t0 = time.time()
                _, res = _process_one_file((filename, input_dir, extractor, scorer, output_dir, settings))
                _record(filename, res, i, len(todo), time.time() - t0)

        if verbose:
            print(f"Terminé en {time.time() - start_all:.1f}s.")
    elif verbose:
        print("Rien à faire : tous les PDF ont déjà été traités (voir la sauvegarde).")

    return [results_by_name[f] for f in pdf_files if f in results_by_name]


### `report.py`

Génère le fichier Excel global (onglets Résumé + Détail_pages, avec les 2 méthodes côte à côte).

In [ ]:
%%writefile report.py
"""
report.py
---------
Génère le fichier Excel global récapitulant, pour chaque PDF traité, les
pages jugées pertinentes selon CHAQUE méthode utilisée (voir
`settings["methods"]` dans config.py) : "density_only" (densité numérique
seule) et/ou "keyword_and_density" (mot-clé obligatoire + densité).

Si les deux méthodes sont actives, elles sont affichées CÔTE À CÔTE dans les
deux feuilles, pour pouvoir comparer directement où elles sont d'accord et
où elles divergent.

Deux feuilles sont produites :
- "Résumé"       : une ligne par PDF (nb pages pertinentes par méthode, etc.)
- "Détail_pages" : une ligne par page traitée (score, mots-clés trouvés,
                    pertinente ou non PAR méthode) -> utile pour auditer /
                    ajuster les seuils et comparer les deux méthodes.
"""

from __future__ import annotations

from typing import List

import pandas as pd

from pdf_processor import PdfProcessingResult

METHOD_LABELS = {
    "density_only": "Densité seule",
    "keyword_and_density": "Mot-clé + densité",
}


def _label(method: str) -> str:
    return METHOD_LABELS.get(method, method)


def _format_pages_list(pages: List[int]) -> str:
    return ", ".join(str(p) for p in pages) if pages else ""


def _collect_methods(results: List[PdfProcessingResult]) -> List[str]:
    """Récupère la liste des méthodes réellement présentes dans les
    résultats, dans un ordre stable (density_only puis keyword_and_density
    en priorité si présentes, puis toute autre méthode rencontrée)."""
    seen = []
    preferred_order = ["density_only", "keyword_and_density"]
    all_methods = set()
    for r in results:
        all_methods.update(r.relevant_pages_by_method.keys())
    for m in preferred_order:
        if m in all_methods:
            seen.append(m)
    for m in sorted(all_methods - set(seen)):
        seen.append(m)
    return seen


def build_summary_dataframe(results: List[PdfProcessingResult]) -> pd.DataFrame:
    methods = _collect_methods(results)
    rows = []
    for r in results:
        row = {
            "Nom du PDF": r.pdf_name,
            "Chemin source": r.input_path,
            "Nombre de pages total": r.total_pages,
        }
        for method in methods:
            pages = r.relevant_pages_by_method.get(method, [])
            row[f"Nb pages pertinentes ({_label(method)})"] = len(pages)
            row[f"Pages pertinentes ({_label(method)})"] = _format_pages_list(pages)
            row[f"Chemin PDF filtré ({_label(method)})"] = r.output_paths_by_method.get(method, "")

        if len(methods) == 2:
            set_a = set(r.relevant_pages_by_method.get(methods[0], []))
            set_b = set(r.relevant_pages_by_method.get(methods[1], []))
            diff = sorted(set_a ^ set_b)
            row["Pages où les méthodes diffèrent"] = _format_pages_list(diff)

        row["Statut"] = "Erreur" if r.error else (
            "Aucune page pertinente" if not any(r.relevant_pages_by_method.values()) else "OK"
        )
        row["Erreur"] = r.error or ""
        rows.append(row)
    return pd.DataFrame(rows)


def build_detail_dataframe(results: List[PdfProcessingResult]) -> pd.DataFrame:
    methods = _collect_methods(results)
    rows = []
    for r in results:
        for pr in r.page_results:
            row = {
                "Nom du PDF": r.pdf_name,
                "Page": pr.page_number,
                "Score composite": round(pr.score, 3),
                "Mots-clés trouvés": ", ".join(pr.matched_keywords),
                "Méthode d'extraction": pr.extraction_method,
            }
            for method in methods:
                row[f"Pertinente ({_label(method)})"] = pr.method_results.get(method, False)
                row[f"Continuation ({_label(method)})"] = pr.continuation_by_method.get(method, False)

            if len(methods) == 2:
                row["Méthodes en désaccord ?"] = pr.method_results.get(
                    methods[0], False
                ) != pr.method_results.get(methods[1], False)

            rows.append(row)
    return pd.DataFrame(rows)


def write_excel_report(results: List[PdfProcessingResult], excel_path: str) -> None:
    summary_df = build_summary_dataframe(results)
    detail_df = build_detail_dataframe(results)

    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        summary_df.to_excel(writer, sheet_name="Résumé", index=False)
        detail_df.to_excel(writer, sheet_name="Détail_pages", index=False)

        # ajustement simple de la largeur des colonnes pour la lisibilité
        for sheet_name, df in [("Résumé", summary_df), ("Détail_pages", detail_df)]:
            ws = writer.sheets[sheet_name]
            for idx, col in enumerate(df.columns, start=1):
                max_len = max(
                    [len(str(col))] + [len(str(v)) for v in df[col].astype(str)]
                )
                ws.column_dimensions[ws.cell(row=1, column=idx).column_letter].width = min(
                    max(max_len + 2, 12), 60
                )

    print(f"Rapport Excel écrit: {excel_path}")


### `density_tester.py`

Outil de test unitaire : affiche pour chaque page la densité + le résultat des 2 méthodes, pour ajuster les seuils.

In [ ]:
%%writefile density_tester.py
"""
density_tester.py
------------------
Outil de test unitaire pour calibrer les seuils de pertinence sur un PDF,
page par page.

Réutilise EXACTEMENT le même code d'extraction et de scoring que le
pipeline principal (`text_extraction.build_extractor`,
`relevance.build_default_composite_scorer`) pour que les résultats observés
ici correspondent à ce qui se passerait réellement pendant le traitement
complet - y compris les DEUX méthodes de décision ("density_only" et
"keyword_and_density"), affichées côte à côte pour comparaison.

Usage typique (dans un notebook) :

    from density_tester import analyze_pdf_density
    import config

    settings = dict(config.DEFAULT_SETTINGS)
    df = analyze_pdf_density("mon_document.pdf", settings)
    df   # affiche un tableau : page, méthode d'extraction, nombre de
         # chiffres, densité, mots-clés trouvés, pertinente pour chaque
         # méthode

Pour tester un extracteur personnalisé (ex: votre propre classe PaddleOCR
héritant de `TextExtractor`), passez-le directement :

    from mon_extracteur_paddle import PaddleOCRExtractor
    mon_extractor = PaddleOCRExtractor(...)
    df = analyze_pdf_density("mon_document.pdf", settings, extractor=mon_extractor)

Ça permet d'ajuster `numeric_density_threshold`, `numeric_min_digit_count`
et `confirmation_threshold` dans `config.py` en observant l'effet réel sur
vos documents, sans relancer tout le pipeline de filtrage à chaque essai.
"""

from __future__ import annotations

from typing import Dict, Optional

import pandas as pd

from text_extraction import build_extractor, TextExtractor
from relevance import build_default_composite_scorer, METHOD_DENSITY_ONLY, METHOD_KEYWORD_AND_DENSITY

try:
    from config import DEFAULT_SETTINGS
except ImportError:
    DEFAULT_SETTINGS = {}

METHOD_LABELS = {
    METHOD_DENSITY_ONLY: "Pertinente (densité seule)",
    METHOD_KEYWORD_AND_DENSITY: "Pertinente (mot-clé + densité)",
}


def analyze_pdf_density(
    pdf_path: str,
    settings: Optional[Dict] = None,
    extractor: Optional[TextExtractor] = None,
) -> pd.DataFrame:
    """Extrait le texte de chaque page de `pdf_path` et évalue sa
    pertinence avec le même CompositeScorer que le pipeline principal.

    Retourne un DataFrame avec une ligne par page :
    - Page, Méthode d'extraction, Longueur texte
    - Nombre de chiffres, Densité (ratio), Score densité (0-1)
    - Mots-clés trouvés
    - Pertinente (densité seule), Pertinente (mot-clé + densité)

    `extractor` : si fourni, utilisé tel quel (utile pour tester votre
    propre implémentation de `TextExtractor`, ex: un extracteur PaddleOCR).
    Sinon, construit automatiquement via `build_extractor(settings)` selon
    `settings["extraction_mode"]`.
    """
    merged = dict(DEFAULT_SETTINGS)
    if settings:
        merged.update(settings)

    if extractor is None:
        extractor = build_extractor(merged)

    scorer = build_default_composite_scorer(merged)
    numeric_scorer = scorer.numeric_scorer  # même instance/réglages que le pipeline

    page_texts = extractor.extract_page_texts(pdf_path)

    rows = []
    for pt in page_texts:
        evaluation = scorer.evaluate(pt.text)
        digit_count = numeric_scorer.digit_count(pt.text)
        density = numeric_scorer.density(pt.text)

        row = {
            "Page": pt.page_number,
            "Méthode d'extraction": pt.method,
            "Longueur texte": len(pt.text),
            "Nombre de chiffres": digit_count,
            "Densité (ratio)": round(density, 4),
            "Score densité (0-1)": round(evaluation.details["numeric_density_score"], 3),
            "Mots-clés trouvés": ", ".join(evaluation.matched_keywords),
        }
        for method, label in METHOD_LABELS.items():
            row[label] = evaluation.method_results.get(method, False)

        rows.append(row)

    return pd.DataFrame(rows)


def print_density_report(
    pdf_path: str,
    settings: Optional[Dict] = None,
    extractor: Optional[TextExtractor] = None,
) -> None:
    """Affiche le rapport dans la console (pratique hors notebook)."""
    df = analyze_pdf_density(pdf_path, settings, extractor=extractor)
    print(df.to_string(index=False))


if __name__ == "__main__":
    import sys

    if len(sys.argv) < 2:
        print("Usage: python density_tester.py chemin/vers/document.pdf")
        sys.exit(1)

    print_density_report(sys.argv[1])


---
### Vérification

Exécutez la cellule ci-dessous pour vérifier que les 5 fichiers ont bien été
écrits dans le dossier courant.

In [ ]:
import os
expected = ["config.py", "text_extraction.py", "relevance.py", "pdf_processor.py", "report.py", "density_tester.py"]
for f in expected:
    status = "OK" if os.path.exists(f) else "MANQUANT"
    print(f"{f:<25} {status}")
